# System RAG – podsumowanie

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Wczytanie tekstu

In [ ]:
from langchain_community.document_loaders import TextLoader

In [ ]:
loader = TextLoader("documents/pan-tadeusz.txt")
documents = loader.load()

## Podział na chunki

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". "],
    chunk_size=1000,
    chunk_overlap=500
)

In [ ]:
chunks = splitter.split_text(documents[0].page_content)

## Embeddingi i zapis do bazy wektorowej

In [ ]:
import time
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import models, QdrantClient
from langchain_qdrant import QdrantVectorStore
from qdrant_client.models import VectorParams, Distance

In [ ]:
client = QdrantClient(url="http://localhost:6333")
model_name = "sdadas/st-polish-paraphrase-from-distilroberta"
collection_name = "pan_tadeusz_podsumowanie"

In [ ]:
client.create_collection(
    collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

In [ ]:
embeddings = HuggingFaceEmbeddings(model=model_name)

In [ ]:
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

In [ ]:
start = time.perf_counter()
vectorstore.add_texts(chunks,
                      metadatas=[{"page_content_len": len(chunk)}
                                 for chunk in chunks],
                     ids=list(range(len(chunks))))
print("Czas:", f"{time.perf_counter() - start:.2f}")

In [ ]:
# start = time.perf_counter()
# vectors = embeddings.embed_documents(chunks)
# print("Czas:", f"{time.perf_counter() - start:.2f}")

## Retrieval i generacja

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank

### Prosty retriever

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
template = """Pozyskane dokumenty:
---------
{context}

-----
Koniec pozyskanych dokumentów
============

Odpowiedz na następujące pytanie: {input}.

=============

Ważna informacja: Możesz odpowiadać wyłącznie na podstawie dostarczonego kontekstu.
Jeśli nie ma w nim informacji na temat zadanego pytania odpowiedz 'Nie wiem'."""

prompt_template = PromptTemplate.from_template(template)

In [ ]:
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
query = "Czym jest żółta księga?"
response = retrieval_chain.invoke({"input": query})

In [ ]:
response["input"]

In [ ]:
response["answer"]

In [ ]:
response["context"]

### Filtrowanie payloadu

**Podejście 1**

In [ ]:
from qdrant_client import models

retriever = vectorstore.as_retriever()
document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)


retriever.search_kwargs = {
    "k": 2,
    "filter": models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.page_content_len",
                range=models.Range(gt=600),
            )
        ]
    ),
}

response = retrieval_chain.invoke({"input": query})

In [ ]:
len(response["context"])

In [ ]:
response["answer"]

**Podejście 2**

In [ ]:
from operator import itemgetter
from langchain_core.runnables import ConfigurableField


base_retriever = vectorstore.as_retriever().configurable_fields(
    search_kwargs=ConfigurableField(
        id="search_kwargs",
        name="Search Kwargs",
        description="Dynamiczne parametry wyszukiwania",
    )
)
retriever = itemgetter("input") | base_retriever

document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

response = retrieval_chain.invoke(
    {"input": query},
    config={
        "configurable": {
            "search_kwargs": {
                "k": 3,
                "filter": models.Filter(
                    must=[
                        models.FieldCondition(
                            key="metadata.page_content_len",
                            range=models.Range(gt=100),
                        )
                    ]
                ),
            }
        }
    },
)

In [ ]:
len(response["context"])

In [ ]:
response["answer"]

### Reranking

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
reranker = LLMListwiseRerank.from_llm(llm=llm, top_n=3)

In [ ]:
retriever_with_reranker = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

In [ ]:
retrieval_chain = create_retrieval_chain(retriever_with_reranker, document_chain)

In [ ]:
response = retrieval_chain.invoke({"input": query})

In [ ]:
response["input"]

In [ ]:
response["answer"]

In [ ]:
context_docs = [doc.page_content for doc in response["context"]]
context_docs

### Hybrid search

In [ ]:
from langchain_qdrant import FastEmbedSparse
from qdrant_client.models import (
    VectorParams, Distance, SparseVectorParams, PointStruct, SparseVector, Prefetch, FusionQuery, Fusion,
)

#### Sparse model (BM25 via FastEmbed)

In [ ]:
sparse_model = FastEmbedSparse(model_name="Qdrant/bm25")

In [ ]:
hybrid_collection = "pan_tadeusz_hybrid"

client.recreate_collection(
    hybrid_collection,
    vectors_config={"dense": VectorParams(size=768, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams()},
)

In [ ]:
dense_vectors = embeddings.embed_documents(chunks)
sparse_vectors = sparse_model.embed_documents(chunks)

points = [
    PointStruct(
        id=i,
        vector={
            "dense": dense_vectors[i],
            "sparse": SparseVector(
                indices=sparse_vectors[i].indices,
                values=sparse_vectors[i].values,
            ),
        },
        payload={"page_content": chunks[i], "metadata": {"page_content_len": len(chunks[i]) }}
    )
    for i in range(len(chunks))
]

client.upsert(collection_name=hybrid_collection, points=points)
f"Dodano {len(points)} chunków"

#### Wyszukiwanie: tylko dense, tylko sparse, hybrid (RRF)

In [ ]:
query = "Czym jest żółta księga?"
k = 3

# Dense – wyszukiwanie semantyczne
results_dense = client.query_points(
    collection_name=hybrid_collection,
    query=embeddings.embed_query(query),
    using="dense",
    limit=k,
).points
results_dense

In [ ]:
# Sparse – BM25 (słowa kluczowe)
sparse_q = sparse_model.embed_query(query)

results_sparse = client.query_points(
    collection_name=hybrid_collection,
    query=SparseVector(indices=sparse_q.indices, values=sparse_q.values),
    using="sparse",
    limit=k,
).points
results_sparse

In [ ]:
# Hybrid – Reciprocal Rank Fusion (dense + sparse)
sparse_q = sparse_model.embed_query(query)

results_hybrid = client.query_points(
    collection_name=hybrid_collection,
    prefetch=[
        Prefetch(query=embeddings.embed_query(query), using="dense", limit=k * 2),
        Prefetch(
            query=SparseVector(indices=sparse_q.indices, values=sparse_q.values),
            using="sparse",
            limit=k * 2,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=k,
).points
results_hybrid

#### Hybrid retriever (LangChain)

In [ ]:
from langchain_qdrant import RetrievalMode

vectorstore_hybrid = QdrantVectorStore(
    client=client,
    collection_name=hybrid_collection,
    embedding=embeddings,
    sparse_embedding=sparse_model,
    retrieval_mode=RetrievalMode.HYBRID,  # RetrievalMode.HYBRID, RetrievalMode.SPARSE, RetrievalMode.DENSE
    vector_name="dense",
    sparse_vector_name="sparse",
)

In [ ]:
start = time.perf_counter()
vectorstore.add_texts(chunks,
                      metadatas=[{"page_content_len": len(chunk)}
                                 for chunk in chunks],
                     ids=list(range(len(chunks))))
print("Czas:", f"{time.perf_counter() - start:.2f}")

In [ ]:
retriever = vectorstore_hybrid.as_retriever(search_kwargs={"k": 3})

document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

response = retrieval_chain.invoke({"input": query})
response["answer"]